# Horizon distance from SNR threshold 6

This notebook computes a representative horizon distance by evaluating the SNR over a grid of chirp masses and luminosity distances, then extracting the distance where SNR = 6.

## Setup

- Detector: ET (single detector)
- Waveform: TaylorF2 (GWFish implementation)
- Representative sources: equal-mass binaries built from selected chirp masses
- Horizon threshold: SNR = 6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import GWFish.modules as gw
from GWFish.modules.horizon import compute_SNR
from GWFish.modules.auxiliary import from_mChirp_q_to_m1_m2

In [ ]:
# Configuration
detector = gw.detection.Detector('ET')
waveform_model = 'TaylorF2'
waveform_class = gw.waveforms.TaylorF2
snr_threshold = 6.0

chirp_masses = np.array([1.2, 2.0, 5.0, 10.0, 20.0, 30.0, 50.0])
luminosity_distances = np.geomspace(10.0, 5.0e4, 80)  # Mpc

# Representative extrinsic parameters (kept fixed for this scan)
extrinsic = {
    'theta_jn': np.pi / 3,
    'ra': 1.0,
    'dec': 0.5,
    'psi': 0.0,
    'phase': 0.0,
    'geocent_time': 1187008882.0,
}


In [ ]:
snr_grid = np.zeros((len(chirp_masses), len(luminosity_distances)))

for i, mchirp in enumerate(chirp_masses):
    m1, m2 = from_mChirp_q_to_m1_m2(mchirp, q=1.0)

    for j, d_l in enumerate(luminosity_distances):
        params = {
            **extrinsic,
            'mass_1_source': float(m1),
            'mass_2_source': float(m2),
            'luminosity_distance': float(d_l),
            'redshift': 0.0,
        }

        snr_grid[i, j] = compute_SNR(
            params=params,
            detector=detector,
            waveform_model=waveform_model,
            waveform_class=waveform_class,
            redefine_tf_vectors=False,
        )

snr_grid

In [ ]:
def horizon_from_snr_curve(distances, snr_values, threshold=6.0):
    distances = np.asarray(distances, dtype=float)
    snr_values = np.asarray(snr_values, dtype=float)

    # Find first crossing from above to below threshold
    above = snr_values >= threshold
    if not np.any(above):
        return np.nan
    if np.all(above):
        return distances[-1]

    idx_below = np.where(~above)[0][0]
    idx_above = idx_below - 1

    d1, d2 = distances[idx_above], distances[idx_below]
    s1, s2 = snr_values[idx_above], snr_values[idx_below]

    # Log-log interpolation
    ld = np.interp(np.log10(threshold), [np.log10(s2), np.log10(s1)], [np.log10(d2), np.log10(d1)])
    return 10 ** ld

horizon_distances = np.array([
    horizon_from_snr_curve(luminosity_distances, snr_grid[i, :], threshold=snr_threshold)
    for i in range(len(chirp_masses))
])

horizon_table = pd.DataFrame({
    'chirp_mass_msun': chirp_masses,
    'horizon_distance_mpc_snr6': horizon_distances,
})
horizon_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

im = ax.pcolormesh(
    luminosity_distances,
    chirp_masses,
    snr_grid,
    shading='auto',
    cmap='viridis',
)
cbar = fig.colorbar(im, ax=ax, label='SNR')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Luminosity distance [Mpc]')
ax.set_ylabel('Chirp mass [M$_\odot$]')
ax.set_title('SNR grid for representative sources')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.loglog(chirp_masses, horizon_distances, marker='o')
plt.xlabel('Chirp mass [M$_\odot$]')
plt.ylabel('Horizon distance [Mpc]')
plt.title('Horizon distance at SNR = 6')
plt.grid(True, which='both', ls=':')
plt.tight_layout()
plt.show()